# Inspeção Visual de Peças Metálicas

Mini-projeto de Machine Learning e Visão Computacional desenvolvido para o programa SCTEC.

## 1. Configuração do ambiente

Nesta etapa, verificamos o ambiente Python, importamos as bibliotecas utilizadas na análise exploratória e definimos os caminhos para acessar o dataset.

In [2]:
import sys

print(f"Python em uso: {sys.executable}")
print(f"Versão do Python: {sys.version}")

Python em uso: c:\Users\USUARIO\Documents\Git-arquivos\sctec-inspecao-qualidade-cnn\.venv\Scripts\python.exe
Versão do Python: 3.13.12 (main, Feb 12 2026, 00:38:53) [MSC v.1944 64 bit (AMD64)]


In [3]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

In [4]:
# Identifica a pasta em que o notebook está sendo executado.
RAIZ_PROJETO = Path.cwd()

# Caso a execução comece em "notebooks", retorna à raiz do projeto.
if RAIZ_PROJETO.name == "notebooks":
    RAIZ_PROJETO = RAIZ_PROJETO.parent

# Define os caminhos do dataset e de suas duas classes.
PASTA_DATASET = RAIZ_PROJETO / "data" / "raw" / "casting_512x512"
PASTA_DEFEITUOSAS = PASTA_DATASET / "def_front"
PASTA_OK = PASTA_DATASET / "ok_front"

# Confere se as pastas foram localizadas.
print(f"Raiz do projeto: {RAIZ_PROJETO}")
print(f"Dataset encontrado: {PASTA_DATASET.exists()}")
print(f"Pasta de peças defeituosas: {PASTA_DEFEITUOSAS.exists()}")
print(f"Pasta de peças OK: {PASTA_OK.exists()}")

Raiz do projeto: c:\Users\USUARIO\Documents\Git-arquivos\sctec-inspecao-qualidade-cnn
Dataset encontrado: True
Pasta de peças defeituosas: True
Pasta de peças OK: True


## 2. Conhecendo o dataset

O dataset está dividido em duas classes:

- `def_front`: peças com defeitos;
- `ok_front`: peças sem defeitos.

A seguir, verificamos a quantidade de imagens e a proporção de cada classe.

In [5]:
# Localiza e organiza os arquivos JPEG de cada classe.
imagens_defeituosas = sorted(PASTA_DEFEITUOSAS.glob("*.jpeg"))
imagens_ok = sorted(PASTA_OK.glob("*.jpeg"))

quantidade_defeituosas = len(imagens_defeituosas)
quantidade_ok = len(imagens_ok)
total_imagens = quantidade_defeituosas + quantidade_ok

# Calcula a participação percentual de cada classe.
percentual_defeituosas = quantidade_defeituosas / total_imagens * 100
percentual_ok = quantidade_ok / total_imagens * 100

print(
    f"Peças defeituosas: {quantidade_defeituosas} "
    f"({percentual_defeituosas:.1f}%)"
)
print(
    f"Peças OK: {quantidade_ok} "
    f"({percentual_ok:.1f}%)"
)
print(f"Total de imagens: {total_imagens}")

Peças defeituosas: 781 (60.1%)
Peças OK: 519 (39.9%)
Total de imagens: 1300


### Análise da distribuição

O dataset contém 1.300 imagens, sendo 781 peças defeituosas (60,1%) e 519 peças sem defeito (39,9%).

Existe um desbalanceamento moderado entre as classes. Portanto, além da acurácia, será importante observar métricas como precisão, recall, F1-score e matriz de confusão durante a avaliação do modelo.

## 3. Validação técnica das imagens

Antes do processamento, verificamos se todas as imagens podem ser carregadas corretamente e se possuem dimensões, número de canais e tipo de dados consistentes.


In [6]:
todas_as_imagens = imagens_defeituosas + imagens_ok

imagens_invalidas = []
dimensoes_encontradas = set()
quantidades_canais = set()
tipos_de_dados = set()

# Percorre todas as imagens para verificar sua integridade e características.
for caminho_imagem in todas_as_imagens:
    imagem = cv2.imread(str(caminho_imagem))

    if imagem is None:
        imagens_invalidas.append(caminho_imagem.name)
        continue

    altura, largura = imagem.shape[:2]
    dimensoes_encontradas.add((largura, altura))

    # Imagens coloridas carregadas pelo OpenCV possuem três canais: B, G e R.
    canais = imagem.shape[2] if imagem.ndim == 3 else 1
    quantidades_canais.add(canais)

    tipos_de_dados.add(str(imagem.dtype))

print(f"Imagens verificadas: {len(todas_as_imagens)}")
print(f"Imagens inválidas: {len(imagens_invalidas)}")
print(f"Dimensões encontradas: {sorted(dimensoes_encontradas)}")
print(f"Quantidade de canais: {sorted(quantidades_canais)}")
print(f"Tipos de dados: {sorted(tipos_de_dados)}")

Imagens verificadas: 1300
Imagens inválidas: 0
Dimensões encontradas: [(512, 512)]
Quantidade de canais: [3]
Tipos de dados: ['uint8']


### Resultado da validação

Todas as 1.300 imagens foram carregadas corretamente, sem arquivos inválidos. As imagens possuem dimensões uniformes de 512 × 512 pixels e três canais de cores.

O tipo `uint8` indica que cada canal utiliza valores inteiros entre 0 e 255. Essa consistência facilita a aplicação dos filtros do OpenCV e o posterior treinamento da CNN.